In [1]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load dataset và xem thông tin tổng quan
df = pd.read_csv('../data/all_recipes_final.csv')
print(f"Dataset shape: {df.shape}")
print(f"Phân bố theo nguồn:")
print(df['source'].value_counts())
df.head()


Dataset shape: (10335, 12)
Phân bố theo nguồn:
source
dienmayxanh    8993
vnexpress       806
vncooking       536
Name: count, dtype: int64


,title,type_of_food,link,description,ingredients,step,note,num_of_ingredients,cook_time,num_of_people,calories,source
0,Cách muối dưa hành truyền thống,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,[],5,45 phút,8-10 người,459 kcal,vnexpress
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,['Su hào xào mực cùng với canh măng mực là hai...,6,50 phút,4 - 5 người,1.162 kcal,vnexpress
2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",['Nếu tận dụng nước luộc gà nấu canh măng thì ...,6,100 phút,8 - 10 người,4.930 kcal,vnexpress
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,['Hạnh nhân xào (hay giả hạnh nhân) là món ăn ...,8,60 phút,4-5 người,1.112 kcal,vnexpress
4,Chả bì ớt xiêm xanh,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",['Nên sơ chế kỹ bì lợn để chả được thơm. Tùy t...,6,60 phút,5-6 người,2.512 kcal,vnexpress


In [3]:
# Xem thông tin chi tiết về các cột
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10335 entries, 0 to 10334
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   title               10335 non-null  object
 1   type_of_food        10335 non-null  object
 2   link                10335 non-null  object
 3   description         10328 non-null  object
 4   ingredients         10335 non-null  object
 5   step                10335 non-null  object
 6   note                10335 non-null  object
 7   num_of_ingredients  10335 non-null  int64 
 8   cook_time           9976 non-null   object
 9   num_of_people       9975 non-null   object
 10  calories            438 non-null    object
 11  source              10335 non-null  object
dtypes: int64(1), object(11)
memory usage: 969.0+ KB


## 1. Data Preprocessing

In [4]:
# Xử lý missing values
data = df.copy()

data['title'] = data['title'].fillna('')
data['description'] = data['description'].fillna('')
data['step'] = data['step'].fillna('[]')
data['ingredients'] = data['ingredients'].fillna('[]')
data['type_of_food'] = data['type_of_food'].fillna('Unknown')

print("Missing values after handling:")
print(data[['title', 'description', 'step', 'ingredients', 'type_of_food', 'calories', 'cook_time']].isnull().sum())


Missing values after handling:
title              0
description        0
step               0
ingredients        0
type_of_food       0
calories        9897
cook_time        359
dtype: int64


In [5]:
# Định nghĩa các hàm xử lý và làm sạch dữ liệu
def parse_list_string(s):
    if pd.isna(s) or s == '[]':
        return []
    try:
        return ast.literal_eval(s)
    except:
        return []

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^\w\s\u00C0-\u1EF9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_cook_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).lower()
    minutes = 0
    
    hour_match = re.search(r'(\d+)\s*(?:giờ|h|hour)', time_str)
    if hour_match:
        minutes += int(hour_match.group(1)) * 60
    
    min_match = re.search(r'(\d+)\s*(?:phút|p|min|minute)', time_str)
    if min_match:
        minutes += int(min_match.group(1))
    
    if minutes == 0:
        num_match = re.search(r'(\d+)', time_str)
        if num_match:
            minutes = int(num_match.group(1))
    
    return minutes if minutes > 0 else np.nan

def parse_calories(cal_str):
    if pd.isna(cal_str):
        return np.nan
    cal_str = str(cal_str).replace('.', '').replace(',', '')
    match = re.search(r'(\d+)', cal_str)
    if match:
        return float(match.group(1))
    return np.nan


In [6]:
# Áp dụng các hàm preprocessing lên dữ liệu
data['ingredients_list'] = data['ingredients'].apply(parse_list_string)
data['step_list'] = data['step'].apply(parse_list_string)

data['title_clean'] = data['title'].apply(clean_text)
data['description_clean'] = data['description'].apply(clean_text)
data['step_clean'] = data['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))

data['cook_time_minutes'] = data['cook_time'].apply(parse_cook_time)
data['calories_numeric'] = data['calories'].apply(parse_calories)

data['ingredients_clean'] = data['ingredients_list'].apply(
    lambda x: set([clean_text(ing) for ing in x if ing])
)

print("✅ Preprocessing completed!")
data[['title', 'title_clean', 'cook_time', 'cook_time_minutes', 'calories', 'calories_numeric']].head()


✅ Preprocessing completed!


,title,title_clean,cook_time,cook_time_minutes,calories,calories_numeric
0,Cách muối dưa hành truyền thống,cách muối dưa hành truyền thống,45 phút,45.0,459 kcal,459.0
1,Su hào xào mực - món cổ Tết Bát Tràng,su hào xào mực món cổ tết bát tràng,50 phút,50.0,1.162 kcal,1162.0
2,Canh măng ngày Tết cổ truyền Hà Nội,canh măng ngày tết cổ truyền hà nội,100 phút,100.0,4.930 kcal,4930.0
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,giả hạnh nhân món ngon tết xưa hà nội,60 phút,60.0,1.112 kcal,1112.0
4,Chả bì ớt xiêm xanh,chả bì ớt xiêm xanh,60 phút,60.0,2.512 kcal,2512.0


In [7]:
# Kiểm tra kết quả preprocessing
print("Sample ingredients_clean:")
for i, ing in enumerate(data['ingredients_clean'].head(3)):
    print(f"\nRecipe {i+1}: {data['title'].iloc[i]}")
    print(f"Ingredients: {ing}")


Sample ingredients_clean:

Recipe 1: Cách muối dưa hành truyền thống
Ingredients: {'lọ sạch', '1 kg hành củ tươi', 'tro bếp hoặc nước vo gọa', 'muối hạt đường', 'cà rốt trang trí tùy chọn'}

Recipe 2: Su hào xào mực - món cổ Tết Bát Tràng
Ingredients: {'1 2 củ cà rốt', '2 củ su hào non', 'gia vị mắm muối đường hạt tiêu rượu trắng gừng', '1 con mực khô', 'mỡ lợn hoặc dầu ăn', 'rau mùi trang trí'}

Recipe 3: Canh măng ngày Tết cổ truyền Hà Nội
Ingredients: {'nước vo gạo ngâm măng', 'nước dùng gà hoặc ninh xương lợn', '2 móng giò lợn', 'gia vị nước mắm truyền thống muối', 'hành khô hành củ', '800 gr măng khô'}


## 2. TF-IDF Based Recommendation

Tính toán similarity dựa trên nội dung văn bản (title, description, steps) sử dụng TF-IDF và cosine similarity.


In [8]:
# Tạo TF-IDF matrix từ text features
data['combined_text'] = data['title_clean'] + ' ' + data['description_clean'] + ' ' + data['step_clean']

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

tfidf_matrix = tfidf_vectorizer.fit_transform(data['combined_text'])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")


TF-IDF Matrix shape: (10335, 5000)


In [9]:
# Tính cosine similarity matrix từ TF-IDF
tfidf_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"TF-IDF Similarity Matrix shape: {tfidf_similarity.shape}")


TF-IDF Similarity Matrix shape: (10335, 10335)


In [10]:
# Hàm lấy recommendations dựa trên TF-IDF
def get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['tfidf_score'] = scores
    
    return result

def recommend_by_title_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}")
    print("\nTF-IDF Recommendations:")
    
    return get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)


## 3. Ingredient-Based Recommendation (Jaccard Similarity)

Tính toán độ tương đồng dựa trên nguyên liệu sử dụng Jaccard Similarity: J(A,B) = |A ∩ B| / |A ∪ B|


In [11]:
# Hàm tính Jaccard similarity giữa 2 sets
def jaccard_similarity(set1, set2):
    if len(set1) == 0 and len(set2) == 0:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0.0

def compute_jaccard_similarity_matrix(ingredients_list):
    n = len(ingredients_list)
    similarity_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i, n):
            sim = jaccard_similarity(ingredients_list[i], ingredients_list[j])
            similarity_matrix[i][j] = sim
            similarity_matrix[j][i] = sim
    
    return similarity_matrix


In [12]:
# Tính Jaccard similarity matrix cho tất cả recipes
print("Computing Jaccard similarity matrix...")
ingredients_sets = data['ingredients_clean'].tolist()
jaccard_similarity_matrix = compute_jaccard_similarity_matrix(ingredients_sets)
print(f"Jaccard Similarity Matrix shape: {jaccard_similarity_matrix.shape}")


Computing Jaccard similarity matrix...
Jaccard Similarity Matrix shape: (10335, 10335)


In [13]:
# Hàm lấy recommendations dựa trên ingredients
def get_ingredient_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['jaccard_score'] = scores
    
    return result

def recommend_by_title_jaccard(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Ingredients: {list(df.loc[recipe_idx, 'ingredients_clean'])[:5]}...")
    print("\nIngredient-Based Recommendations:")
    
    return get_ingredient_recommendations(recipe_idx, similarity_matrix, df, top_n)


## 4. Metadata Similarity

Tính toán similarity dựa trên metadata: calories, cook_time, type_of_food


In [14]:
# Hàm tính similarity dựa trên metadata (calories, cook_time, type_of_food)
def compute_metadata_similarity_matrix(df):
    scaler = MinMaxScaler()
    
    calories = df['calories_numeric'].fillna(df['calories_numeric'].median()).values.reshape(-1, 1)
    cook_time = df['cook_time_minutes'].fillna(df['cook_time_minutes'].median()).values.reshape(-1, 1)
    
    calories_normalized = scaler.fit_transform(calories)
    cook_time_normalized = scaler.fit_transform(cook_time)
    
    type_dummies = pd.get_dummies(df['type_of_food'], prefix='type')
    
    metadata_features = np.hstack([
        calories_normalized,
        cook_time_normalized,
        type_dummies.values
    ])
    
    similarity_matrix = cosine_similarity(metadata_features, metadata_features)
    
    return similarity_matrix, metadata_features


In [15]:
# Tính metadata similarity matrix
print("Computing Metadata similarity matrix...")
metadata_similarity_matrix, metadata_features = compute_metadata_similarity_matrix(data)
print(f"✅ Metadata Similarity Matrix shape: {metadata_similarity_matrix.shape}")


Computing Metadata similarity matrix...


✅ Metadata Similarity Matrix shape: (10335, 10335)


In [16]:
# Hàm lấy recommendations dựa trên metadata
def get_metadata_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['metadata_score'] = scores
    
    return result

def recommend_by_title_metadata(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"❌ No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\n🍽️ Input: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}, Time: {df.loc[recipe_idx, 'cook_time']}")
    print("\n📊 Metadata-Based Recommendations:")
    
    return get_metadata_recommendations(recipe_idx, similarity_matrix, df, top_n)


## 5. Hybrid Approach

Kết hợp 3 phương pháp với trọng số: TF-IDF (0.3) + Jaccard (0.5) + Metadata (0.2)


**Rationale**: Ingredients quan trọng nhất trong food recommendation, nên tăng Jaccard weight

In [17]:
# Hàm tính hybrid similarity (kết hợp 3 phương pháp)
def compute_hybrid_similarity(tfidf_sim, jaccard_sim, metadata_sim, 
                              w_tfidf=0.3, w_jaccard=0.5, w_metadata=0.2):
    total_weight = w_tfidf + w_jaccard + w_metadata
    w_tfidf /= total_weight
    w_jaccard /= total_weight
    w_metadata /= total_weight
    
    print(f"Weights: TF-IDF={w_tfidf:.2f}, Jaccard={w_jaccard:.2f}, Metadata={w_metadata:.2f}")
    
    hybrid_similarity = (w_tfidf * tfidf_sim + 
                        w_jaccard * jaccard_sim + 
                        w_metadata * metadata_sim)
    
    return hybrid_similarity


In [18]:
# Tính hybrid similarity matrix
print("Computing Hybrid similarity matrix...")
hybrid_similarity_matrix = compute_hybrid_similarity(
    tfidf_similarity, 
    jaccard_similarity_matrix, 
    metadata_similarity_matrix,
    w_tfidf=0.3,
    w_jaccard=0.5,
    w_metadata=0.2
)
print(f"✅ Hybrid Similarity Matrix shape: {hybrid_similarity_matrix.shape}")


Computing Hybrid similarity matrix...
Weights: TF-IDF=0.30, Jaccard=0.50, Metadata=0.20
✅ Hybrid Similarity Matrix shape: (10335, 10335)


In [19]:
# Hàm lấy recommendations dựa trên hybrid approach
def get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, jaccard_sim, metadata_sim, df, top_n=5):
    sim_scores = list(enumerate(hybrid_sim[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    hybrid_scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['hybrid_score'] = hybrid_scores
    result['tfidf_score'] = [tfidf_sim[recipe_idx][i] for i in recipe_indices]
    result['jaccard_score'] = [jaccard_sim[recipe_idx][i] for i in recipe_indices]
    result['metadata_score'] = [metadata_sim[recipe_idx][i] for i in recipe_indices]
    
    return result

def recommend_by_title_hybrid(title, df, hybrid_sim, tfidf_sim, jaccard_sim, metadata_sim, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"❌ No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\n🍽️ Input: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}, Time: {df.loc[recipe_idx, 'cook_time']}")
    print("\n📊 Hybrid Recommendations:")
    
    return get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, jaccard_sim, metadata_sim, df, top_n)


## 6. Ingredient TF-IDF Based Recommendation

**Ingredient TF-IDF** xử lý ingredient list như text documents, tốt hơn Jaccard vì:
- Xử lý được variations trong cách viết (thịt bò, bò, beef...)
- Gán trọng số cho ingredients based on importance
- Không bị ảnh hưởng bởi exact string matching

**Ý tưởng**: Mỗi recipe là 1 document, ingredients là words. Áp dụng TF-IDF để tính similarity.

In [20]:
# Chuẩn bị ingredient text cho TF-IDF
# Join ingredients thành string separated by spaces
data['ingredients_text'] = data['ingredients_list'].apply(
    lambda x: ' '.join([clean_text(ing) for ing in x if ing])
)

print("Sample ingredient texts:")
for i in range(3):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Ingredients text: {data['ingredients_text'].iloc[i][:100]}...")

# Build TF-IDF vectorizer cho ingredients
print("\nBuilding Ingredient TF-IDF matrix...")
ingredient_tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,  # Fewer features than text-based
    ngram_range=(1, 2),  # Unigrams and bigrams
    min_df=2,
    max_df=0.8
)

ingredient_tfidf_matrix = ingredient_tfidf_vectorizer.fit_transform(data['ingredients_text'])
print(f"✅ Ingredient TF-IDF Matrix shape: {ingredient_tfidf_matrix.shape}")

# Tính cosine similarity
ingredient_tfidf_similarity = cosine_similarity(ingredient_tfidf_matrix, ingredient_tfidf_matrix)
print(f"✅ Ingredient TF-IDF Similarity Matrix shape: {ingredient_tfidf_similarity.shape}")

Sample ingredient texts:

Cách muối dưa hành truyền thống
   Ingredients text: 1 kg hành củ tươi tro bếp hoặc nước vo gọa muối hạt đường cà rốt trang trí tùy chọn lọ sạch...

Su hào xào mực - món cổ Tết Bát Tràng
   Ingredients text: 2 củ su hào non 1 con mực khô 1 2 củ cà rốt gia vị mắm muối đường hạt tiêu rượu trắng gừng rau mùi t...

Canh măng ngày Tết cổ truyền Hà Nội
   Ingredients text: 800 gr măng khô 2 móng giò lợn nước dùng gà hoặc ninh xương lợn hành khô hành củ gia vị nước mắm tru...

Building Ingredient TF-IDF matrix...
✅ Ingredient TF-IDF Matrix shape: (10335, 2000)
✅ Ingredient TF-IDF Similarity Matrix shape: (10335, 10335)


In [21]:
# Hàm lấy recommendations dựa trên Ingredient TF-IDF
def get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['ing_tfidf_score'] = scores
    
    return result

def recommend_by_title_ingredient_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\n🍽️ Input: {df.loc[recipe_idx, 'title']}")
    print(f"   Ingredients: {df.loc[recipe_idx, 'ingredients_text'][:100]}...")
    print("\n🥘 Ingredient TF-IDF Recommendations:")
    
    return get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)

print("Ingredient TF-IDF recommendation functions ready!")

Ingredient TF-IDF recommendation functions ready!


In [22]:
# Test Ingredient TF-IDF Recommendation
test_recipe = "Thịt bò xào"
display(recommend_by_title_ingredient_tfidf(test_recipe, data, ingredient_tfidf_similarity, top_n=10))


🍽️ Input: Thịt bò xào hoa thiên lý nhanh gọn, bổ dưỡng cho ngày hè
   Ingredients: 300 gr hoa thiên lý 200 gr thịt bò 3 4 tép tỏi ớt tùy chọn gia vị mắm dầu hào hạt nêm hạt tiêu dầu ă...

🥘 Ingredient TF-IDF Recommendations:


,title,type_of_food,calories,cook_time,ing_tfidf_score
111,Bò thuôn hành răm món 'quốc dân' đậm vị Hà Nội,Món ngon hàng ngày,820 kcal,20 phút,0.541460
107,Niễng xào thịt bò - đặc sản ''trời ban'' vào đông,Món ngon hàng ngày,735 kcal,25 phút,0.540380
100,Tép đồng rang ba chỉ,Món ngon hàng ngày,871 kcal,30 phút,0.497931
774,"Canh ghẹ nấu rau muống đơn giản, thanh mát",Thực đơn cho ngày nắng nóng,818 kcal,30 phút,0.452315
168,"Tôm rang thịt ba chỉ măn ngọt, béo ngậy ngon cơm",Món ngon hàng ngày,1.058 kcal,35 phút,0.431621
124,Ếch xào măng - món chân quê vào nhà hàng,Món ngon hàng ngày,940 kcal,35 phút,0.431176
116,Bò xào sả ớt mềm ngon chỉ trong 15 phút,Món ngon hàng ngày,860 kcal,15 phút,0.421437
767,Bầu xào trứng - món đơn giản mà đưa cơm ngày hè,Thực đơn cho ngày nắng nóng,643 kcal,15 phút,0.420113
109,Tim heo xào chua ngọt,Món ngon hàng ngày,883 kcal,35 phút,0.409063
401,Rạm kho lá lốt – món ngon Thái Bình,Món ngon hàng ngày,708 kcal,40 phút,0.401328


## 7. Keyword-Based Recommendation

Extract keywords chính từ title (nguyên liệu chính + phương pháp nấu).

**Ưu điểm**: Đơn giản, focus vào main keywords, dễ giải thích

In [23]:
# Hàm lấy recommendations dựa trên keywords
def get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['keyword_score'] = scores
    
    return result

def recommend_by_title_keyword(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\n🍽️ Input: {df.loc[recipe_idx, 'title']}")
    print(f"   Keywords: {df.loc[recipe_idx, 'keywords']}")
    print("\nKeyword-Based Recommendations:")
    
    return get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n)

print("✅ Keyword-based functions ready!")

✅ Keyword-based functions ready!


In [24]:
# Hàm extract keywords/tags từ title
def extract_keywords(title):
    """
    Extract main keywords from recipe title
    Focus on: ingredients, cooking methods, dish types
    """
    if pd.isna(title):
        return set()
    
    title = clean_text(title)
    
    # Common Vietnamese cooking methods and dish types
    cooking_methods = ['xào', 'nướng', 'luộc', 'chiên', 'hấp', 'kho', 'rim', 'rang', 
                       'canh', 'súp', 'cháo', 'gỏi', 'nộm', 'salad', 'bún', 'phở', 
                       'mì', 'cơm', 'bánh', 'chè', 'sinh tố']
    
    # Common ingredients
    main_ingredients = ['thịt', 'bò', 'gà', 'heo', 'lợn', 'cá', 'tôm', 'mực', 'nghêu',
                        'rau', 'củ', 'quả', 'trứng', 'đậu', 'nấm', 'măng', 'bí', 
                        'cà', 'khoai', 'su', 'hào', 'cải', 'rau muống', 'rau cần']
    
    # Extract keywords
    keywords = set()
    words = title.split()
    
    # Add cooking methods
    for method in cooking_methods:
        if method in title:
            keywords.add(method)
    
    # Add ingredients (check for 2-word and 1-word matches)
    for i in range(len(words)):
        # Check 2-word combinations
        if i < len(words) - 1:
            two_word = f"{words[i]} {words[i+1]}"
            for ing in main_ingredients:
                if ing in two_word:
                    keywords.add(ing)
        
        # Check single words
        for ing in main_ingredients:
            if ing in words[i]:
                keywords.add(ing)
    
    # Add all significant words (length > 2) as backup
    for word in words:
        if len(word) > 2:
            keywords.add(word)
    
    return keywords

# Apply keyword extraction to all recipes
print("Extracting keywords from all recipes...")
data['keywords'] = data['title'].apply(extract_keywords)
print(f"✅ Keyword extraction completed!")

# Show examples
print("\nSample keywords:")
for i in range(5):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Keywords: {data['keywords'].iloc[i]}")

Extracting keywords from all recipes...
✅ Keyword extraction completed!

Sample keywords:

Cách muối dưa hành truyền thống
   Keywords: {'cách', 'dưa', 'truyền', 'cá', 'muối', 'thống', 'hành'}

Su hào xào mực - món cổ Tết Bát Tràng
   Keywords: {'su', 'hào', 'tràng', 'xào', 'bát', 'mực', 'món', 'tết'}

Canh măng ngày Tết cổ truyền Hà Nội
   Keywords: {'ngày', 'truyền', 'măng', 'nội', 'canh', 'gà', 'tết'}

Giả hạnh nhân - món ngon Tết xưa Hà Nội
   Keywords: {'nhân', 'nội', 'xưa', 'tết', 'ngon', 'hạnh', 'món', 'giả'}

Chả bì ớt xiêm xanh
   Keywords: {'xanh', 'xiêm', 'chả'}


In [25]:
# Tính Keyword-based similarity matrix
print("Computing Keyword similarity matrix...")
keywords_sets = data['keywords'].tolist()
keyword_similarity_matrix = compute_jaccard_similarity_matrix(keywords_sets)
print(f"✅ Keyword Similarity Matrix shape: {keyword_similarity_matrix.shape}")

Computing Keyword similarity matrix...
✅ Keyword Similarity Matrix shape: (10335, 10335)


In [26]:
# So sánh Jaccard vs Ingredient TF-IDF vs Keyword
test_recipe = "canh chua"

print("="*100)
print("JACCARD (Original Ingredient-Based)")
print("="*100)
display(recommend_by_title_jaccard(test_recipe, data, jaccard_similarity_matrix, top_n=5))

print("\n" + "="*100)
print("INGREDIENT TF-IDF (NEW - Better Ingredient Matching)")
print("="*100)
display(recommend_by_title_ingredient_tfidf(test_recipe, data, ingredient_tfidf_similarity, top_n=5))

print("\n" + "="*100)
print("KEYWORD-BASED")
print("="*100)
display(recommend_by_title_keyword(test_recipe, data, keyword_similarity_matrix, top_n=5))

JACCARD (Original Ingredient-Based)

Input: Canh chua cá khoai mềm, ngọt tự nhiên
   Ingredients: ['3 củ hành khô', 'dầu ăn', '1 2 quả dứa', '3 quả cà chua', '1 nhánh gừng nhỏ']...

Ingredient-Based Recommendations:


,title,type_of_food,calories,cook_time,jaccard_score
41,Su hào xào mực - đặc sản Bát Tràng,Món Tết,1.091 kcal,50 phút,0.214286
153,Cá trích kho cà chua giá rẻ mà tốn cơm ngày đông,Món ngon hàng ngày,1.516 kcal,120 phút,0.153846
133,Tàu hủ khìa nước dừa đơn giản mà hao cơm,Món ngon hàng ngày,905 kcal,45 phút,0.142857
83,Lòng thuôn hành răm - Món đưa cơm ngày nồm ẩm,Món ngon hàng ngày,1.261 kcal,30 phút,0.133333
116,Bò xào sả ớt mềm ngon chỉ trong 15 phút,Món ngon hàng ngày,860 kcal,15 phút,0.133333



INGREDIENT TF-IDF (NEW - Better Ingredient Matching)

🍽️ Input: Canh chua cá khoai mềm, ngọt tự nhiên
   Ingredients: 400 gr cá khoai 3 quả cà chua 1 2 quả dứa 1 2 quả chanh 3 củ hành khô 1 nhánh gừng nhỏ gia vị mắm mu...

🥘 Ingredient TF-IDF Recommendations:


,title,type_of_food,calories,cook_time,ing_tfidf_score
104,Nấu canh cá dọc mùng ấm ngày se lạnh,Món ngon hàng ngày,1.170 kcal,45 phút,0.549958
495,Cá thu sốt cà chua - món ăn quốc dân cho ngày ...,Món ngon ngày lạnh,1.174 kcal,40 phút,0.526533
504,"Bắp cải cuốn thịt mềm ngon, đậm vị",Món ngon ngày lạnh,1.116 kcal,50 phút,0.521316
501,Canh cá nấu su hào ngọt ấm ngày lạnh,Món ngon ngày lạnh,1.169 kcal,45 phút,0.508335
438,Cách nầu canh moi nấu dưa chua - món đưa cơm n...,Món ngon hàng ngày,478 kcal,30 phút,0.500634



KEYWORD-BASED

🍽️ Input: Canh chua cá khoai mềm, ngọt tự nhiên
   Keywords: {'khoai', 'cá', 'ngọt', 'mềm', 'canh', 'nhiên', 'chua', 'kho'}

Keyword-Based Recommendations:


,title,type_of_food,calories,cook_time,keyword_score
249,Canh chua cá trê,Món ngon hàng ngày,NaN,NaN,0.333333
643,Canh chua cá Nam bộ,Món ngon theo vùng miền,NaN,NaN,0.333333
1012,Canh chua cá lóc,Món chính,NaN,30phút,0.333333
1669,"Canh cá khoai rau cải ngọt ngon, dinh dưỡng ch...",Món canh,NaN,15 phút,0.333333
1512,Canh cá khoai cải cúc (tần ô) ngọt mát đưa cơm...,Món canh,NaN,15 phút,0.312500


In [27]:
# Class FoodRecommender hoàn chỉnh - với Ingredient TF-IDF, Keyword-based (BM25 disabled)
class FoodRecommender:
    
    def __init__(self, df):
        self.df = df.copy()
        self._preprocess()
        self._build_similarity_matrices()
    
    def _preprocess(self):
        self.df['ingredients_list'] = self.df['ingredients'].apply(parse_list_string)
        self.df['step_list'] = self.df['step'].apply(parse_list_string)
        
        self.df['title_clean'] = self.df['title'].apply(clean_text)
        self.df['description_clean'] = self.df['description'].fillna('').apply(clean_text)
        self.df['step_clean'] = self.df['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))
        self.df['combined_text'] = self.df['title_clean'] + ' ' + self.df['description_clean'] + ' ' + self.df['step_clean']
        
        self.df['cook_time_minutes'] = self.df['cook_time'].apply(parse_cook_time)
        self.df['calories_numeric'] = self.df['calories'].apply(parse_calories)
        
        self.df['ingredients_clean'] = self.df['ingredients_list'].apply(
            lambda x: set([clean_text(ing) for ing in x if ing])
        )
        
        # Ingredient text for TF-IDF
        self.df['ingredients_text'] = self.df['ingredients_list'].apply(
            lambda x: ' '.join([clean_text(ing) for ing in x if ing])
        )
        
        # Extract keywords for keyword-based recommendation
        self.df['keywords'] = self.df['title'].apply(extract_keywords)
        
        print("Data preprocessing completed!")
    
    def _build_similarity_matrices(self):
        print("Building TF-IDF similarity...")
        tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=1, max_df=0.95)
        tfidf_matrix = tfidf_vectorizer.fit_transform(self.df['combined_text'])
        self.tfidf_sim = cosine_similarity(tfidf_matrix)
        
        # BM25 DISABLED - Too slow
        # print("Building BM25 similarity...")
        # bm25_corpus = [text.split() for text in self.df['combined_text']]
        # bm25_model = BM25Okapi(bm25_corpus)
        # self.bm25_sim = compute_bm25_similarity_matrix(bm25_model, bm25_corpus)
        
        print("Building Jaccard similarity...")
        self.jaccard_sim = compute_jaccard_similarity_matrix(self.df['ingredients_clean'].tolist())
        
        print("Building Ingredient TF-IDF similarity...")
        ing_tfidf_vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=2, max_df=0.8)
        ing_tfidf_matrix = ing_tfidf_vectorizer.fit_transform(self.df['ingredients_text'])
        self.ing_tfidf_sim = cosine_similarity(ing_tfidf_matrix)
        
        print("Building Keyword similarity...")
        self.keyword_sim = compute_jaccard_similarity_matrix(self.df['keywords'].tolist())
        
        print("Building Metadata similarity...")
        self.metadata_sim, _ = compute_metadata_similarity_matrix(self.df)
        
        print("Building Hybrid similarity...")
        self.hybrid_sim = compute_hybrid_similarity(
            self.tfidf_sim, self.jaccard_sim, self.metadata_sim,
            w_tfidf=0.3, w_jaccard=0.5, w_metadata=0.2
        )
        
        print("All similarity matrices ready!")
    
    def recommend(self, title, method='hybrid', top_n=5):
        """
        Recommend recipes based on title
        
        Args:
            method: 'tfidf', 'jaccard', 'ing_tfidf', 'keyword', 'hybrid'
            method: 'tfidf', 'bm25', 'jaccard', 'ing_tfidf', 'keyword', 'hybrid'
            top_n: Number of recommendations
        """
        matches = self.df[self.df['title'].str.contains(title, case=False, na=False)]
        if len(matches) == 0:
            print(f"No recipe found with title containing: {title}")
            return None
        
        recipe_idx = matches.index[0]
        print(f"\n🍽️ Input: {self.df.loc[recipe_idx, 'title']}")
        
        return self.recommend_by_index(recipe_idx, method=method, top_n=top_n)
    
    def recommend_by_index(self, idx, method='hybrid', top_n=5):
        """
        Recommend recipes by index
        
        Args:
            method: 'tfidf', 'jaccard', 'ing_tfidf', 'keyword', 'hybrid'
            method: 'tfidf', 'bm25', 'jaccard', 'ing_tfidf', 'keyword', 'hybrid'
            top_n: Number of recommendations
        """
        if idx < 0 or idx >= len(self.df):
            print(f"Invalid index: {idx}")
            return None
        
        if method == 'tfidf':
            return get_tfidf_recommendations(idx, self.tfidf_sim, self.df, top_n)
        elif method == 'jaccard':
            return get_ingredient_recommendations(idx, self.jaccard_sim, self.df, top_n)
        elif method == 'ing_tfidf':
            return get_ingredient_tfidf_recommendations(idx, self.ing_tfidf_sim, self.df, top_n)
        elif method == 'keyword':
            return get_keyword_recommendations(idx, self.keyword_sim, self.df, top_n)
        elif method == 'hybrid':
            return get_hybrid_recommendations(
                idx, self.hybrid_sim, self.tfidf_sim,
                self.jaccard_sim, self.metadata_sim, self.df, top_n
            )
        else:
            print(f"Unknown method: {method}")
            return None

print("✅ FoodRecommender class updated!")

✅ FoodRecommender class updated!


In [28]:
# Khởi tạo recommender instance (5 methods: TF-IDF, Jaccard, Ing TF-IDF, Keyword, Hybrid)
print("Initializing Food Recommender with 5 methods...")
recommender_v3 = FoodRecommender(df)

Initializing Food Recommender with 5 methods...
Data preprocessing completed!
Building TF-IDF similarity...
Building Jaccard similarity...
Building Ingredient TF-IDF similarity...
Building Keyword similarity...
Building Metadata similarity...
Building Hybrid similarity...
Weights: TF-IDF=0.30, Jaccard=0.50, Metadata=0.20
All similarity matrices ready!


## 8. Testing All Methods - Comprehensive Comparison

In [41]:
# So sánh TẤT CẢ phương pháp với cùng 1 món
test_recipe = "cơm chiên"

print("="*100)
print("📝 TF-IDF METHOD (Text-based)")
print("="*100)
display(recommender_v3.recommend(test_recipe, method='tfidf', top_n=5))

print("\n" + "="*100)
print("🥘 JACCARD METHOD (Original Ingredient-Based)")
print("="*100)
display(recommender_v3.recommend(test_recipe, method='jaccard', top_n=5))

print("\n" + "="*100)
print("🍲 INGREDIENT TF-IDF METHOD (NEW - Better Ingredient Matching)")
print("="*100)
display(recommender_v3.recommend(test_recipe, method='ing_tfidf', top_n=5))

print("\n" + "="*100)
print("🏷️  KEYWORD-BASED METHOD")
print("="*100)
display(recommender_v3.recommend(test_recipe, method='keyword', top_n=5))

print("\n" + "="*100)
print("🎯 HYBRID METHOD (TF-IDF + Jaccard + Metadata)")
print("="*100)
display(recommender_v3.recommend(test_recipe, method='hybrid', top_n=5))

📝 TF-IDF METHOD (Text-based)

🍽️ Input: Cơm chiên Dương Châu hải sản


,title,type_of_food,calories,cook_time,tfidf_score
5047,"Cháo hải sản nhiều dinh dưỡng, ấm bụng cực đơn...",Món cháo,NaN,45 phút,0.525237
5304,Cơm tay cầm hải sản thơm ngon hấp dẫn dễ làm c...,Món xào,NaN,25 phút,0.475161
4236,Bánh canh phồng tôm dẻo dai bằng nồi inox cho ...,Món nước,NaN,25 phút,0.460223
6878,Nem hải sản chay thơm ngon giòn rụm cho ngày Rằm,Món chiên,NaN,10 phút,0.446877
4837,Hải sản phô mai đút lò đơn giản cực ngon ai cũ...,Món nướng,NaN,50 phút,0.445070



🥘 JACCARD METHOD (Original Ingredient-Based)

🍽️ Input: Cơm chiên Dương Châu hải sản


,title,type_of_food,calories,cook_time,jaccard_score
1133,Bún riêu cua tại nhà,Món chính,NaN,60phút,0.105263
825,"Trứng onsen kiểu Nhật đơn giản, béo ngậy chỉ v...",Món chính,NaN,15phút,0.090909
700,Ba bước làm bánh khoai lang tím 'núng nính,Quà - Món ăn vặt,NaN,30 phút,0.083333
1067,Rau muống xào tỏi nhanh chóng,Món chay,NaN,15phút,0.083333
1103,Cá điêu hồng chiên tỏi,Món chính,NaN,30phút,0.083333



🍲 INGREDIENT TF-IDF METHOD (NEW - Better Ingredient Matching)

🍽️ Input: Cơm chiên Dương Châu hải sản


,title,type_of_food,calories,cook_time,ing_tfidf_score
1192,Trứng chiên thịt heo bằm,Món chính,NaN,15phút,0.431065
7998,Xôi lạp xưởng ngon dẻo bằng lò vi sóng siêu nh...,Món hấp,NaN,30 phút,0.416855
5880,Mì xào singapore thơm ngon mới lạ đổi vị cho b...,Món xào,NaN,30 phút,0.376028
4233,"Món súp bánh phồng tôm đơn giản, thơm ngon lạ ...",Món nước,NaN,30 phút,0.355866
5769,Nui xào hải sản sốt me lạ miệng tươi mát đơn giản,Món xào,NaN,30 phút,0.351457



🏷️  KEYWORD-BASED METHOD

🍽️ Input: Cơm chiên Dương Châu hải sản


,title,type_of_food,calories,cook_time,keyword_score
1303,Cơm chiên dương châu chay vẫn ngon miệng,Món chay,NaN,45phút,0.400000
7194,Cơm chiên cà ri hải sản thơm ngon hấp dẫn,Món chiên,NaN,30 phút,0.363636
535,Hải sản chiên nướng chấm tương chua,Món ngon cho cuối tuần,NaN,NaN,0.300000
633,Cơm chiên lá é - đặc sản Nha Trang,Món ngon theo vùng miền,1.135 kcal,45 phút,0.300000
361,Món lạ hải sản,Món ngon hàng ngày,NaN,NaN,0.285714



🎯 HYBRID METHOD (TF-IDF + Jaccard + Metadata)

🍽️ Input: Cơm chiên Dương Châu hải sản


,title,type_of_food,calories,cook_time,hybrid_score,tfidf_score,jaccard_score,metadata_score
361,Món lạ hải sản,Món ngon hàng ngày,NaN,NaN,0.309968,0.366560,0.0,1.000000
145,Bí quyết làm miến xào hải sản không bị dính,Món ngon hàng ngày,1.366 kcal,30 phút,0.300739,0.335886,0.0,0.999864
489,Hướng dẫn cách làm tôm xóc bơ tỏi thơm ngon AI...,Món ngon hàng ngày,NaN,NaN,0.281496,0.271654,0.0,1.000000
236,Quen lạ với tôm,Món ngon hàng ngày,NaN,NaN,0.280291,0.267636,0.0,1.000000
184,Cơm cháy kho quẹt giòn rụm ăn kèm rau củ quả,Món ngon hàng ngày,1.739 kcal,55 phút,0.278553,0.261944,0.0,0.999849


## 9. Final Summary

### 5 Phương pháp Content-Based Recommendation đã implement:

1. **TF-IDF Based**: Text similarity (title + description + steps) với Cosine similarity
   - Ưu: Tốt cho text matching, phát hiện món ăn có cách nấu tương tự
   - Nhược: Phụ thuộc vào chất lượng text
   - Use case: Tìm món có description/cách nấu giống nhau

2. **Jaccard (Ingredient-Based)**: Ingredient similarity với $J(A,B) = \frac{|A \cap B|}{|A \cup B|}$
   - Ưu: Đơn giản, dễ giải thích
   - ⚠️ Nhược: Không chính xác với variations trong ingredient naming
   - Use case: Baseline ingredient comparison

3. **Ingredient TF-IDF** ⭐ **RECOMMENDED**: TF-IDF trên ingredient text
   - Ưu: Tốt hơn Jaccard, xử lý variations, gán trọng số cho ingredients
   - Ưu: Không bị exact string matching problem
   - **Recommended thay thế Jaccard**
   - Use case: Tìm món có nguyên liệu tương đồng

4. **Keyword/Tag-Based**: Extract keywords từ title (thịt bò, xào, canh...) và Jaccard similarity
   - Ưu: Đơn giản, focus vào main ingredients và cooking methods
   - Ưu: Dễ hiểu và giải thích, nhanh
   - Use case: Quick matching based on key terms

5. **Hybrid**: Weighted combination - TF-IDF (0.3) + Jaccard (0.5) + Metadata (0.2)
   - Ưu: Kết hợp nhiều signals, robust
   - Nhược: Phức tạp, cần tune weights
   - Use case: Best overall personalized recommendation

### Metadata Features:
- calories, cook_time, type_of_food

### So Sánh và Khuyến Nghị:

| Method | Độ chính xác | Speed | Complexity | Best For |
|--------|--------------|-------|------------|----------|
| TF-IDF | ⭐⭐⭐ | Trung bình | Đơn giản | Text/description matching |
| Jaccard | ⭐⭐ | Nhanh | Đơn giản | Baseline (not recommended) |
| **Ing TF-IDF** | ⭐⭐⭐⭐ | Trung bình | Đơn giản | **Best ingredient matching** ✅ |
| Keyword | ⭐⭐⭐ | Nhanh | Đơn giản | Quick keyword matching |
| Hybrid | ⭐⭐⭐⭐ | Chậm | Phức tạp | Overall best ✅ |

### Top 3 Methods Recommended:
1. **Ingredient TF-IDF** ⭐ - Best cho ingredient-based recommendation
2. **Hybrid** ⭐ - Best overall khi kết hợp multiple signals
3. **Keyword** - Nhanh nhất, tốt cho quick search

In [30]:
# Class FoodRecommender hoàn chỉnh - tích hợp tất cả phương pháp
class FoodRecommender:
    
    def __init__(self, df):
        self.df = df.copy()
        self._preprocess()
        self._build_similarity_matrices()
    
    def _preprocess(self):
        self.df['ingredients_list'] = self.df['ingredients'].apply(parse_list_string)
        self.df['step_list'] = self.df['step'].apply(parse_list_string)
        
        self.df['title_clean'] = self.df['title'].apply(clean_text)
        self.df['description_clean'] = self.df['description'].fillna('').apply(clean_text)
        self.df['step_clean'] = self.df['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))
        self.df['combined_text'] = self.df['title_clean'] + ' ' + self.df['description_clean'] + ' ' + self.df['step_clean']
        
        self.df['cook_time_minutes'] = self.df['cook_time'].apply(parse_cook_time)
        self.df['calories_numeric'] = self.df['calories'].apply(parse_calories)
        
        self.df['ingredients_clean'] = self.df['ingredients_list'].apply(
            lambda x: set([clean_text(ing) for ing in x if ing])
        )
        
        print("✅ Data preprocessing completed!")
    
    def _build_similarity_matrices(self):
        print("Building TF-IDF similarity...")
        tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=1, max_df=0.95)
        tfidf_matrix = tfidf_vectorizer.fit_transform(self.df['combined_text'])
        self.tfidf_sim = cosine_similarity(tfidf_matrix)
        
        print("Building Jaccard similarity...")
        self.jaccard_sim = compute_jaccard_similarity_matrix(self.df['ingredients_clean'].tolist())
        
        print("Building Metadata similarity...")
        self.metadata_sim, _ = compute_metadata_similarity_matrix(self.df)
        
        print("Building Hybrid similarity...")
        self.hybrid_sim = compute_hybrid_similarity(
            self.tfidf_sim, self.jaccard_sim, self.metadata_sim,
            w_tfidf=0.3, w_jaccard=0.5, w_metadata=0.2
        )
        
        print("✅ All similarity matrices ready!")
    
    def recommend(self, title, method='hybrid', top_n=5):
        matches = self.df[self.df['title'].str.contains(title, case=False, na=False)]
        if len(matches) == 0:
            print(f"❌ No recipe found with title containing: {title}")
            return None
        
        recipe_idx = matches.index[0]
        print(f"\n🍽️ Input: {self.df.loc[recipe_idx, 'title']}")
        
        if method == 'tfidf':
            return get_tfidf_recommendations(recipe_idx, self.tfidf_sim, self.df, top_n)
        elif method == 'jaccard':
            return get_ingredient_recommendations(recipe_idx, self.jaccard_sim, self.df, top_n)
        else:
            return get_hybrid_recommendations(
                recipe_idx, self.hybrid_sim, self.tfidf_sim, 
                self.jaccard_sim, self.metadata_sim, self.df, top_n
            )
    
    def recommend_by_index(self, idx, method='hybrid', top_n=5):
        if idx < 0 or idx >= len(self.df):
            print(f"❌ Invalid index: {idx}")
            return None
        
        print(f"\n🍽️ Input: {self.df.iloc[idx]['title']}")
        
        if method == 'tfidf':
            return get_tfidf_recommendations(idx, self.tfidf_sim, self.df, top_n)
        elif method == 'jaccard':
            return get_ingredient_recommendations(idx, self.jaccard_sim, self.df, top_n)
        else:
            return get_hybrid_recommendations(
                idx, self.hybrid_sim, self.tfidf_sim,
                self.jaccard_sim, self.metadata_sim, self.df, top_n
            )


In [31]:
# Khởi tạo recommender instance
print("Initializing Food Recommender...")
recommender = FoodRecommender(df)


Initializing Food Recommender...
✅ Data preprocessing completed!
Building TF-IDF similarity...
Building Jaccard similarity...
Building Metadata similarity...
Building Hybrid similarity...
Weights: TF-IDF=0.30, Jaccard=0.50, Metadata=0.20
✅ All similarity matrices ready!


## 10. Export Functions for Evaluation

In [32]:
# Các hàm export dữ liệu cho team Evaluation (5 methods: TF-IDF, Jaccard, Ing TF-IDF, Keyword, Hybrid)
def export_recommendations_for_evaluation(recommender, sample_indices, 
                                         methods=['tfidf', 'jaccard', 'ing_tfidf', 'keyword', 'hybrid'], 
                                         top_n=10):
    results = {method: [] for method in methods}
    
    for idx in sample_indices:
        recipe_title = recommender.df.iloc[idx]['title']
        
        for method in methods:
            recs = recommender.recommend_by_index(idx, method=method, top_n=top_n)
            if recs is not None:
                # Get score column name
                if method == 'ing_tfidf':
                    score_col = 'ing_tfidf_score'
                elif method == 'hybrid':
                    score_col = 'hybrid_score'
                else:
                    score_col = f'{method}_score'
                
                rec_data = {
                    'query_idx': idx,
                    'query_title': recipe_title,
                    'recommendations': recs['title'].tolist(),
                    'scores': recs[score_col].tolist()
                }
                results[method].append(rec_data)
    
    return results

def get_all_similarity_scores(recommender, query_idx, top_n=None):
    n_recipes = len(recommender.df)
    
    results = pd.DataFrame({
        'idx': range(n_recipes),
        'title': recommender.df['title'].values,
        'type_of_food': recommender.df['type_of_food'].values,
        'tfidf_score': recommender.tfidf_sim[query_idx],
        'jaccard_score': recommender.jaccard_sim[query_idx],
        'ing_tfidf_score': recommender.ing_tfidf_sim[query_idx],
        'keyword_score': recommender.keyword_sim[query_idx],
        'metadata_score': recommender.metadata_sim[query_idx],
        'hybrid_score': recommender.hybrid_sim[query_idx]
    })
    
    results = results[results['idx'] != query_idx]
    results = results.sort_values('hybrid_score', ascending=False)
    
    if top_n:
        results = results.head(top_n)
    
    return results.reset_index(drop=True)

def export_to_csv(recommender, output_dir='../evaluation_data'):
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    recipe_data = recommender.df[['title', 'type_of_food', 'calories', 'cook_time', 'source']].copy()
    recipe_data.to_csv(f'{output_dir}/recipes_info.csv', index=True, encoding='utf-8-sig')
    
    np.save(f'{output_dir}/tfidf_similarity.npy', recommender.tfidf_sim)
    np.save(f'{output_dir}/bm25_similarity.npy', recommender.bm25_sim)
    np.save(f'{output_dir}/jaccard_similarity.npy', recommender.jaccard_sim)
    np.save(f'{output_dir}/ing_tfidf_similarity.npy', recommender.ing_tfidf_sim)
    np.save(f'{output_dir}/keyword_similarity.npy', recommender.keyword_sim)
    np.save(f'{output_dir}/metadata_similarity.npy', recommender.metadata_sim)
    np.save(f'{output_dir}/hybrid_similarity.npy', recommender.hybrid_sim)
    
    print(f"✅ Exported all data to: {output_dir}")

print("✅ Export functions ready!")

✅ Export functions ready!


In [33]:
# Test với recipe có nguyên liệu phổ biến hơn
print("📊 Testing with recipe 100 (has common ingredients):")
scores_df = get_all_similarity_scores(recommender_v3, query_idx=100, top_n=10)
display(scores_df)

print("\n" + "="*80)
print("📊 Testing with recipe 0 (unique ingredients - expect low Jaccard):")
scores_df_0 = get_all_similarity_scores(recommender_v3, query_idx=0, top_n=10)
display(scores_df_0)

📊 Testing with recipe 100 (has common ingredients):


,idx,title,type_of_food,tfidf_score,jaccard_score,ing_tfidf_score,keyword_score,metadata_score,hybrid_score
0,168,"Tôm rang thịt ba chỉ măn ngọt, béo ngậy ngon cơm",Món ngon hàng ngày,0.653113,0.250000,0.813446,0.166667,0.999955,0.520925
1,90,Lươn om chuối đậu nóng hổi ngày đông,Món ngon hàng ngày,0.264317,0.142857,0.434420,0.000000,0.998351,0.350394
2,193,Cá kho trám đưa cơm ngày lạnh,Món ngon hàng ngày,0.210124,0.166667,0.434317,0.000000,0.988216,0.344014
3,177,Ba chỉ rang cháy cạnh thơm ngon như ngoài quán,Món ngon hàng ngày,0.460772,0.000000,0.473807,0.181818,0.999362,0.338104
4,284,Tôm rang thịt ba chỉ,Món ngon hàng ngày,0.446694,0.000000,0.182706,0.333333,0.999201,0.333848
5,112,Cá kho trám xanh - thức quà mùa thu,Món ngon hàng ngày,0.309692,0.071429,0.290583,0.000000,0.997315,0.328085
6,174,"Cá trắm kho mật mía đượm vị, không cần nước",Món ngon hàng ngày,0.240272,0.111111,0.303903,0.000000,0.988401,0.325317
7,175,Canh cà bung thịt đậu dân dã mà bổ dưỡng miền Bắc,Món ngon hàng ngày,0.274113,0.071429,0.315421,0.000000,0.999922,0.317933
8,118,"Món cà pháo ram thịt ba chỉ, lá lốt",Món ngon hàng ngày,0.390428,0.000000,0.455624,0.100000,0.999959,0.317120
9,164,Thịt luộc chấm mắm tép 'đưa cơm' ngày hè,Món ngon hàng ngày,0.377043,0.000000,0.355794,0.083333,0.999232,0.312959



📊 Testing with recipe 0 (unique ingredients - expect low Jaccard):


,idx,title,type_of_food,tfidf_score,jaccard_score,ing_tfidf_score,keyword_score,metadata_score,hybrid_score
0,52,"Cách muối hành trắng giòn, để được lâu",Món Tết,0.607932,0.0,0.499080,0.363636,0.998227,0.382025
1,34,Dưa món giòn ngon đón Tết,Món Tết,0.306864,0.0,0.191863,0.083333,0.999364,0.291932
2,53,Dưa góp giòn ngon cho ngày Tết,Món Tết,0.306297,0.0,0.087102,0.071429,0.999937,0.291876
3,30,Nộm tai heo dưa chuột giải ngán ngày Tết,Món Tết,0.287857,0.0,0.067929,0.062500,0.999869,0.286331
4,43,Nộm gà hoa chuối giòn ngon đổi vị ngày Tết,Món Tết,0.287353,0.0,0.143242,0.000000,0.999250,0.286056
5,23,Gà bóp hành răm kiểu miền Trung,Món Tết,0.283008,0.0,0.295931,0.076923,0.999959,0.284894
6,27,Bò cuốn cải xanh giải ngán ngày Tết,Món Tết,0.245385,0.0,0.057530,0.000000,0.999719,0.273559
7,64,Mứt cà rốt không cần nước vôi trong,Món Tết,0.242425,0.0,0.317261,0.000000,0.995879,0.271903
8,33,Tré Huế - đặc sản cố đô vào dịp Tết,Món Tết,0.233899,0.0,0.035480,0.000000,0.997769,0.269723
9,57,Thịt kho tàu kiểu Bắc – món ngon Tết đến,Món Tết,0.220363,0.0,0.063865,0.000000,0.996272,0.265363


In [34]:
# Uncomment để export data cho team Evaluation
# export_to_csv(recommender_v3, output_dir='../evaluation_data')

## 11. Usage Guide (Final Version)

### Sử dụng recommender_v3 (5 Methods):
```python
# 1. TF-IDF method (text-based)
recommender_v3.recommend("Thịt bò", method='tfidf', top_n=5)

# 2. Jaccard (baseline ingredient-based - not recommended)
recommender_v3.recommend("Thịt bò", method='jaccard', top_n=5)

# 3. Ingredient TF-IDF ⭐ RECOMMENDED for ingredients
recommender_v3.recommend("Thịt bò", method='ing_tfidf', top_n=5)

# 4. Keyword-based (fast, simple)
recommender_v3.recommend("Thịt bò xào", method='keyword', top_n=5)

# 5. Hybrid ⭐ RECOMMENDED for best overall
recommender_v3.recommend("Thịt bò", method='hybrid', top_n=5)

# Recommend by index
recommender_v3.recommend_by_index(0, method='ing_tfidf', top_n=5)
```

### Export cho Evaluation:
```python
# Export all similarity matrices (5 methods)
export_to_csv(recommender_v3, output_dir='../evaluation_data')

# Load similarity matrices
tfidf_sim = np.load('../evaluation_data/tfidf_similarity.npy')
jaccard_sim = np.load('../evaluation_data/jaccard_similarity.npy')
ing_tfidf_sim = np.load('../evaluation_data/ing_tfidf_similarity.npy')
keyword_sim = np.load('../evaluation_data/keyword_similarity.npy')
hybrid_sim = np.load('../evaluation_data/hybrid_similarity.npy')
recipes_info = pd.read_csv('../evaluation_data/recipes_info.csv')
```

### Method Selection Guide:

**For ingredient-based recommendation:**
- Use `ing_tfidf` ⭐ (BEST) or `keyword` (faster, simpler)
- Avoid `jaccard` (accuracy issues with Vietnamese text)

**For text-based recommendation:**
- Use `tfidf` (standard baseline)

**For overall best results:**
- Use `hybrid` ⭐ (combines TF-IDF + Jaccard + Metadata)